# Creating Environments
Experimenting with how to set up custom Brax environments.

In [1]:
import jax
from jax import numpy as jp
from matplotlib.lines import Line2D
from matplotlib.patches import Circle
import matplotlib.pyplot as plt
import brax


## Visualizing A Bouncing Ball

In [25]:
from brax.io import mjcf

ball = mjcf.loads(
    """<mujoco>
         <option timestep="0.005"/>
         <worldbody>
           <body pos="0 0 3">
             <joint type="free"/>
             <geom size="0.5" type="sphere"/>
           </body>
           <geom size="40 40 40" type="plane"/>
         </worldbody>
       </mujoco>
  """)

In [26]:
print(ball.link)

Link(transform=Transform(pos=Array([[0., 0., 0.]], dtype=float32), rot=Array([[1., 0., 0., 0.]], dtype=float32)), joint=Transform(pos=Array([[0., 0., 0.]], dtype=float32), rot=Array([[1., 0., 0., 0.]], dtype=float32)), inertia=Inertia(transform=Transform(pos=Array([[0., 0., 0.]], dtype=float32), rot=Array([[1., 0., 0., 0.]], dtype=float32)), i=Array([[[52.35988,  0.     ,  0.     ],
        [ 0.     , 52.35988,  0.     ],
        [ 0.     ,  0.     , 52.35988]]], dtype=float32), mass=Array([523.59875], dtype=float32)), invweight=Array([0.00190986], dtype=float32), constraint_stiffness=Array([2000.], dtype=float32), constraint_vel_damping=Array([0.], dtype=float32), constraint_limit_stiffness=Array([1000.], dtype=float32), constraint_ang_damping=Array([0.], dtype=float32))


__Simulating__

In [27]:
from brax.positional import pipeline

elasticity = 0.85
ball_velocity = 0

# change the material elasticity of the ball and the plane
ball = ball.replace(elasticity=jp.array([elasticity] * ball.ngeom))

# provide an initial velocity to the ball
qd = jp.array([ball_velocity, 0, 0, 0, 0, 0])
state = jax.jit(pipeline.init)(ball, ball.init_q, qd)

states = [state]
for i in range(1000):
  state = jax.jit(pipeline.step)(ball, state, None)
  states.append(state)
  if i % 100 == 0:
      print(state)


State(q=Array([0.       , 0.       , 2.9997547, 1.       , 0.       , 0.       ,
       0.       ], dtype=float32), qd=Array([ 0.        ,  0.        , -0.04906654,  0.        ,  0.        ,
        0.        ], dtype=float32), x=Transform(pos=Array([[0.       , 0.       , 2.9997547]], dtype=float32), rot=Array([[1., 0., 0., 0.]], dtype=float32)), xd=Motion(ang=Array([[0., 0., 0.]], dtype=float32), vel=Array([[ 0.        ,  0.        , -0.04906654]], dtype=float32)), contact=None, x_i=Transform(pos=Array([[0.       , 0.       , 2.9997547]], dtype=float32), rot=Array([[1., 0., 0., 0.]], dtype=float32)), xd_i=Motion(ang=Array([[0., 0., 0.]], dtype=float32), vel=Array([[ 0.        ,  0.        , -0.04906654]], dtype=float32)), j=Transform(pos=Array([[0.       , 0.       , 2.9997547]], dtype=float32), rot=Array([[1., 0., 0., 0.]], dtype=float32)), jd=Motion(ang=Array([[0., 0., 0.]], dtype=float32), vel=Array([[ 0.        ,  0.        , -0.04906654]], dtype=float32)), a_p=Transform(pos=Arra

__Saving And Visualizing__

In [36]:
from brax.io import json as brax_json
from pathlib import Path
from datetime import datetime

def save_brax_trajectory(name: str, system, trajectory_states):
    trajectory_dir = Path('trajectories')
    trajectory_dir.mkdir(exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

    trajectory_path = f'{trajectory_dir}/{name}_{timestamp}.json'
    brax_json.save(trajectory_path, system, trajectory_states)

In [37]:
save_brax_trajectory("ball", ball, states)

Now run `python brax/visualizer/visualizer.py`, go to `localhost:8080` in the browser and check if the visualization server is running.

If it runs, you can play back the simulation using `/play`. For example:

```
localhost:8080/play/path_to_json.json
```

## Other Environments
### Three Ball

In [39]:
three_ball = mjcf.loads(
    """<mujoco>
         <option timestep="0.005"/>
         <worldbody>
           <body pos="0 0 3">
             <joint type="free"/>
             <geom size="0.5" type="sphere"/>
           </body>
                      <body pos="0 0 6">
             <joint type="free"/>
             <geom size="0.5" type="sphere"/>
           </body>
          <body pos="0 0 9">
             <joint type="free"/>
             <geom size="0.5" type="sphere"/>
           </body>
           <geom size="40 40 40" type="plane"/>
         </worldbody>
       </mujoco>
  """)

elasticity = 0.85 #@param { type:"slider", min: 0.5, max: 1.0, step:0.05 }
ball_velocity = 0 #@param { type:"slider", min:-5, max:5, step: 0.5 }

# change the material elasticity of the ball and the plane
three_ball = three_ball.replace(elasticity=jp.array([elasticity] * three_ball.ngeom))

# provide an initial velocity to the ball
qd = jp.array([[ball_velocity, 0, 0, 0, 0, 0] * three_ball.ngeom])
state = jax.jit(pipeline.init)(three_ball, three_ball.init_q, qd)

states = [state]
for i in range(1000):
  state = jax.jit(pipeline.step)(three_ball, state, None)
  states.append(state)
  if i % 100 == 0:
      print(state)

save_brax_trajectory("three_ball", three_ball, states)


State(q=Array([0.       , 0.       , 2.9997547, 1.       , 0.       , 0.       ,
       0.       , 0.       , 0.       , 5.999755 , 1.       , 0.       ,
       0.       , 0.       , 0.       , 0.       , 8.999755 , 1.       ,
       0.       , 0.       , 0.       ], dtype=float32), qd=Array([ 0.        ,  0.        , -0.04906654,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        , -0.04901886,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        , -0.04901886,
        0.        ,  0.        ,  0.        ], dtype=float32), x=Transform(pos=Array([[0.       , 0.       , 2.9997547],
       [0.       , 0.       , 5.999755 ],
       [0.       , 0.       , 8.999755 ]], dtype=float32), rot=Array([[1., 0., 0., 0.],
       [1., 0., 0., 0.],
       [1., 0., 0., 0.]], dtype=float32)), xd=Motion(ang=Array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]], dtype=float32), vel=Array([[ 0.        ,  0.        , -0.04906654],
       [ 0.        ,  0.        

### Shapes?

In [48]:
shapes = mjcf.loads(
    """<mujoco>
         <option timestep="0.005"/>
         <worldbody>
            <body pos="0 0 3">
                <joint type="free"/>
                <geom size="0.5" type="sphere"/>
            </body>
            <body pos="0 0 6">
                <joint type="free"/>
                <geom size="0.5 0.5" type="capsule"/>
           </body>
            <body pos="0 0 9">
                <joint type="free"/>
                <geom size="0.5 0.5 0.5" type="box"/>
           </body>
           <geom size="40 40 40" type="plane"/>
         </worldbody>
       </mujoco>
  """)

elasticity = 0.85 #@param { type:"slider", min: 0.5, max: 1.0, step:0.05 }
ball_velocity = 0 #@param { type:"slider", min:-5, max:5, step: 0.5 }

# change the material elasticity of the ball and the plane
shapes = shapes.replace(elasticity=jp.array([elasticity] * shapes.ngeom))

# provide an initial velocity to the ball
qd = jp.array([[ball_velocity, 0, 0, 0, 0, 0] * shapes.ngeom])
state = jax.jit(pipeline.init)(shapes, shapes.init_q, qd)

states = [state]
for i in range(1000):
  state = jax.jit(pipeline.step)(shapes, state, None)
  states.append(state)
  if i % 100 == 0:
      print(state)

save_brax_trajectory("shapes", shapes, states)


State(q=Array([0.       , 0.       , 2.9997547, 1.       , 0.       , 0.       ,
       0.       , 0.       , 0.       , 5.999755 , 1.       , 0.       ,
       0.       , 0.       , 0.       , 0.       , 8.999755 , 1.       ,
       0.       , 0.       , 0.       ], dtype=float32), qd=Array([ 0.        ,  0.        , -0.04906654,  0.        ,  0.        ,
        0.        ,  0.        ,  0.        , -0.04901886,  0.        ,
        0.        ,  0.        ,  0.        ,  0.        , -0.04901886,
        0.        ,  0.        ,  0.        ], dtype=float32), x=Transform(pos=Array([[0.       , 0.       , 2.9997547],
       [0.       , 0.       , 5.999755 ],
       [0.       , 0.       , 8.999755 ]], dtype=float32), rot=Array([[1., 0., 0., 0.],
       [1., 0., 0., 0.],
       [1., 0., 0., 0.]], dtype=float32)), xd=Motion(ang=Array([[0., 0., 0.],
       [0., 0., 0.],
       [0., 0., 0.]], dtype=float32), vel=Array([[ 0.        ,  0.        , -0.04906654],
       [ 0.        ,  0.        

### Bowling!

In [74]:
bowling = mjcf.loads(
"""<mujoco>
<option timestep="0.005"/>
    <worldbody>
        <body pos="0.001 20 0.7">
            <joint type="free"/>
            <geom size="0.7" type="sphere" rgba="0 0 0 0.8" mass="30000"/>
        </body>
        <body pos="0 0 1">
            <joint type="free"/>
            <geom size="0.5 0.5" type="capsule"/>
        </body>
        <body pos="1.1 0 1">
            <joint type="free"/>
            <geom size="0.5 0.5" type="capsule"/>
        </body>
        <body pos="-1.1 0 1">
            <joint type="free"/>
            <geom size="0.5 0.5" type="capsule"/>
        </body>
        <body pos="-2.2 0 1">
            <joint type="free"/>
            <geom size="0.5 0.5" type="capsule"/>
        </body>
        <body pos="2.2 0 1">
            <joint type="free"/>
            <geom size="0.5 0.5" type="capsule"/>
        </body>
        <body pos="0 1.1 1">
            <joint type="free"/>
            <geom size="0.5 0.5" type="capsule" rgba="1 0.584 0 1"/>
        </body>
        <body pos="1.3 1.1 1">
            <joint type="free"/>
            <geom size="0.5 0.5" type="capsule"/>
        </body>
        <body pos="-1.3 1.1 1">
            <joint type="free"/>
            <geom size="0.5 0.5" type="capsule"/>
        </body>
        <body pos="0.7 2.2 1">
            <joint type="free"/>
            <geom size="0.5 0.5" type="capsule"/>
        </body>
        <body pos="-0.7 2.2 1">
            <joint type="free"/>
            <geom size="0.5 0.5" type="capsule"/>
        </body>
        <body pos="0 3.3 1">
            <joint type="free"/>
            <geom size="0.5 0.5" type="capsule"/>
        </body>
    <geom size="50 50 50" type="plane" rgba="1 0.8 0.8 1"/>
    </worldbody>
    </mujoco>
""")

elasticity = 0.85 #@param { type:"slider", min: 0.5, max: 1.0, step:0.05 }
ball_velocity = 60 #@param { type:"slider", min:-5, max:5, step: 0.5 }

# change the material elasticity of the ball and the plane
bowling = bowling.replace(elasticity=jp.array([elasticity] * bowling.ngeom))

# provide an initial velocity to the ball
qd = jp.array([[0, 0, 0, 0, 0, 0] * bowling.ngeom])
qd = qd.at[0,1].set(-ball_velocity)
state = jax.jit(pipeline.init)(bowling, bowling.init_q, qd)

states = [state]
for i in range(1000):
  state = jax.jit(pipeline.step)(bowling, state, None)
  states.append(state)
  if i % 100 == 0:
      print(state)

save_brax_trajectory("bowling", bowling, states)


State(q=Array([ 1.0000000e-03,  1.9700001e+01,  6.9999284e-01,  1.0000000e+00,
        0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
        0.0000000e+00,  9.9999970e-01,  1.0000000e+00,  0.0000000e+00,
        0.0000000e+00,  0.0000000e+00,  1.1000000e+00,  0.0000000e+00,
        9.9999970e-01,  1.0000000e+00,  0.0000000e+00,  0.0000000e+00,
        0.0000000e+00, -1.1000000e+00,  0.0000000e+00,  9.9999970e-01,
        1.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00,
       -2.2000000e+00,  0.0000000e+00,  9.9999970e-01,  1.0000000e+00,
        0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  2.2000000e+00,
        0.0000000e+00,  9.9999970e-01,  1.0000000e+00,  0.0000000e+00,
        0.0000000e+00,  0.0000000e+00,  0.0000000e+00,  1.1000000e+00,
        9.9999970e-01,  1.0000000e+00,  0.0000000e+00,  0.0000000e+00,
        0.0000000e+00,  1.3000000e+00,  1.1000000e+00,  9.9999970e-01,
        1.0000000e+00,  0.0000000e+00,  0.0000000e+00,  0.0000000e+00